# 06 — Three-Region Constant-Field Model  
**Experiment:** Approximating the SPAD's electric field as three constant-field slabs to get closed-form ionization, multiplication, and trigger estimates.

The full simulation gives $E(x)$ from the Poisson solver — a continuous profile across 7 layers. But the physics can be understood with a much simpler picture:

| Region | Where | Field | Role |
|--------|-------|-------|------|
| 1 — Dead | p+ cap, buffer, substrate | $E \approx 0$ | No ionization, just drift |
| 2 — Multiplication | i-InP (0.5 µm) | $E_{\text{mult}} \sim 4.5\times10^5$ V/cm | High field → avalanche |
| 3 — Absorption | InGaAs + grading (1.62 µm) | $E_{\text{abs}} \sim 0.8\times10^5$ V/cm | Low field, photon absorption only |

Taking each region's field as **constant** gives analytic formulas instead of solving ODEs numerically.

---
## 1. Imports

In [ ]:
import sys, os, warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# Add project root to path so we can import src.*
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.ingestion import DataIngestionConfig, DataIngestionService
from src.avalanche.ionization import VanOverstraetenDeManModel
from src.core.constants import q, kB

vodm = VanOverstraetenDeManModel()
T = 300.0
print('Imports OK')

---
## 2. Pick representative fields from a full simulation

In [ ]:
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
sim = svc.build_simulator()
device = sim.device

Vbr, _ = sim.find_breakdown(V_start=20, V_max=80, V_step=1.0)
phi, E, _ = sim.solve_poisson(Vbr - 2)
x_um = device.grid.x * 1e4
E_abs = np.abs(E)

# Layer boundaries
cum = 0.0
bounds_um = []
for lyr in device.layers:
    cum += lyr.thickness
    bounds_um.append(cum * 1e4)

mult_start, mult_end = bounds_um[0], bounds_um[1]
abs_start,  abs_end  = bounds_um[2], bounds_um[4]

mask_mult = (x_um >= mult_start) & (x_um < mult_end)
mask_abs  = (x_um >= abs_start) & (x_um < abs_end)

E_mult = np.mean(E_abs[mask_mult]) if np.any(mask_mult) else 4.5e5
E_abs_field = np.mean(E_abs[mask_abs]) if np.any(mask_abs) else 0.8e5

print(f"Vbr = {Vbr:.1f} V")
print(f"Multiplication region ({mult_start:.1f}–{mult_end:.1f} µm): E_mult = {E_mult:.2e} V/cm")
print(f"Absorption region  ({abs_start:.1f}–{abs_end:.1f} µm): E_abs  = {E_abs_field:.2e} V/cm")

plt.figure(figsize=(10, 4))
plt.plot(x_um, E_abs, 'b-', lw=1.5, label='Full simulation E(x)')
E_model = np.zeros_like(x_um)
E_model[mask_mult] = E_mult
E_model[mask_abs]  = E_abs_field
plt.plot(x_um, E_model, 'r--', lw=2.5, label='3-region constant-field model')
for b in bounds_um:
    plt.axvline(b, color='gray', ls=':', alpha=0.3)
plt.xlabel('Depth (µm)')
plt.ylabel('|E| (V/cm)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.yscale('log')
plt.title('Three-region constant-field approximation')
plt.show()

---
## 3. Constant-field ionization coefficients

In [ ]:
def ionization_at(E, T=300.0):
    return vodm.alpha(E, None, T), vodm.beta(E, None, T)

alpha_mult, beta_mult = ionization_at(E_mult)
alpha_abs, beta_abs   = ionization_at(E_abs_field)

print(f"Multiplication region:")
print(f"  alpha = {alpha_mult:.2e} cm⁻¹, beta = {beta_mult:.2e} cm⁻¹, k = {beta_mult/alpha_mult:.4f}")
print(f"Absorption region:")
print(f"  alpha = {alpha_abs:.2e} cm⁻¹, beta = {beta_abs:.2e} cm⁻¹, k = {beta_abs/alpha_abs:.4f}")

In [ ]:
E_range = np.logspace(4.5, 6.2, 200)
alpha_r = np.array([vodm.alpha(e, None, T) for e in E_range])
beta_r  = np.array([vodm.beta(e, None, T) for e in E_range])

plt.figure(figsize=(10, 4))
plt.loglog(E_range, alpha_r, 'b-', lw=2, label='$\\alpha(E)$')
plt.loglog(E_range, beta_r, 'b--', lw=2, label='$\\beta(E)$')
plt.axvline(E_mult, color='r', ls='--', alpha=0.6, label=f'E_mult')
plt.axvline(E_abs_field, color='orange', ls='--', alpha=0.6, label=f'E_abs')
plt.xlabel('Electric Field (V/cm)')
plt.ylabel('Ionization Coefficient (cm⁻¹)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Where the two regions sit on the ionization curve')
plt.show()

---
## 4. Dead-space length

$$l_d = \frac{E_{\text{th}}}{|E|} \quad \text{(constant per region)}$$

In [ ]:
def dead_space(E, Eg=1.35, carrier='electron'):
    Eth = 1.5 * Eg if carrier == 'electron' else 1.0 * Eg
    return Eth / abs(E) if abs(E) > 1e4 else 0.0

ld_me = dead_space(E_mult)
ld_mh = dead_space(E_mult, carrier='hole')

alpha_eff = alpha_mult / (1 + alpha_mult * ld_me)
beta_eff  = beta_mult / (1 + beta_mult * ld_mh)

print(f"l_d(e) = {ld_me*1e4:.3f} µm, l_d(h) = {ld_mh*1e4:.3f} µm")
print(f"alpha_eff = {alpha_eff:.2e} cm⁻¹  (raw: {alpha_mult:.2e})")
print(f"beta_eff  = {beta_eff:.2e} cm⁻¹  (raw: {beta_mult:.2e})")

---
## 5. Closed-form multiplication factor

For constant $\alpha, \beta$ over width $W$ with $\delta = \alpha-\beta$:

$$
M = \frac{1}{1 - \frac{\beta}{\delta}(e^{\delta W} - 1)}
$$

In [ ]:
def mcintyre_M(alpha, beta, W):
    if alpha < 1e-10 and beta < 1e-10:
        return 1.0
    d = alpha - beta
    if abs(d) < 1e-10:
        denom = 1.0 - alpha * W
    else:
        denom = 1.0 - beta * (np.exp(d * W) - 1.0) / d
    if denom <= 0.01:
        return 1e6
    return min(1.0 / denom, 1e6)

M_mult = mcintyre_M(alpha_eff, beta_eff, 0.5e-4)
print(f"M from multiplier alone = {M_mult:.1f}")
print(f"M from absorber alone   = 1.0 (negligible field)")

### 5.1 M vs field — the analytic blow-up

In [ ]:
E_sweep = np.linspace(3.5e5, 5.5e5, 100)
M_sw = [mcintyre_M(*(ionization_at(E) + (0.5e-4,))) for E in E_sweep]

plt.figure(figsize=(8, 4))
plt.semilogy(E_sweep/1e5, M_sw, 'r-', lw=2)
plt.axhline(1e6, color='gray', ls='--', alpha=0.4)
plt.xlabel('Field (×10⁵ V/cm)')
plt.ylabel('M')
plt.grid(alpha=0.3)
plt.title('Closed-form M vs field (W = 0.5 µm)')
plt.show()

---
## 6. Comparison with full simulation

In [ ]:
V_range = np.arange(30, Vbr + 10, 2.0)
M_full, M_model = [], []

for V in V_range:
    phi, E, _ = sim.solve_poisson(float(V))
    E_abs = np.abs(E)
    alpha = sim.ionization.effective_alpha_n(E_abs, Eg=1.35)
    beta  = sim.ionization.effective_alpha_p(E_abs, Eg=1.35)
    x = device.grid.x
    active = E_abs > 1e5
    if not np.any(active):
        M_full.append(1.0)
    else:
        dx = np.diff(x); diff = alpha - beta
        cum = np.zeros_like(x)
        for i in range(len(x)-2, -1, -1):
            cum[i] = cum[i+1] + diff[i] * dx[i]
        integrand = beta * np.exp(cum)
        denom = 1.0 - float(np.trapezoid(integrand, x))
        M_full.append(1e6 if denom <= 0.01 else min(1.0/denom, 1e6))

    E_peak = np.max(E_abs[mask_mult]) if np.any(mask_mult) else 4.5e5
    a, b = ionization_at(E_peak)
    M_model.append(mcintyre_M(a/(1+a*dead_space(E_peak)), b/(1+b*dead_space(E_peak,carrier='hole')), 0.5e-4))

plt.figure(figsize=(8, 4))
plt.semilogy(V_range, M_full, 'bo-', lw=1.5, ms=4, label='Full sim')
plt.semilogy(V_range, M_model, 'r^--', lw=1.5, ms=4, label='3-region model')
plt.axvline(Vbr, color='gray', ls=':', alpha=0.5)
plt.xlabel('Bias (V)'); plt.ylabel('M')
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.title('M: full simulation vs constant-field model')
plt.show()

---
## 7. Trigger probability in constant field

In [ ]:
def trigger_const(alpha, beta, W, N=200):
    x = np.linspace(0, W, N)
    Pe, Ph = np.zeros(N), np.zeros(N)
    for _ in range(100):
        Ptr = Pe + Ph - Pe*Ph
        ie = np.zeros(N)
        for i in range(N-2, -1, -1):
            dx = x[i+1] - x[i]
            ie[i] = ie[i+1] + alpha*(Ptr[i]+Ptr[i+1])*dx/2
        PeN = 1 - np.exp(-ie)
        ih = np.zeros(N)
        for i in range(1, N):
            dx = x[i] - x[i-1]
            ih[i] = ih[i-1] + beta*(Ptr[i]+Ptr[i-1])*dx/2
        PhN = 1 - np.exp(-ih)
        err = np.max(np.abs(Pe-PeN)) + np.max(np.abs(Ph-PhN))
        Pe, Ph = 0.3*PeN + 0.7*Pe, 0.3*PhN + 0.7*Ph
        if err < 1e-8: break
    return x, Pe, Ph

x_tr, Pe_tr, Ph_tr = trigger_const(alpha_eff, beta_eff, 0.5e-4)

plt.figure(figsize=(8, 3))
plt.plot(x_tr*1e4, Pe_tr, 'b-', lw=2, label='$P_e$')
plt.plot(x_tr*1e4, Ph_tr, 'r-', lw=2, label='$P_h$')
plt.plot(x_tr*1e4, Pe_tr+Ph_tr-Pe_tr*Ph_tr, 'g-', lw=2, label='$P_{\\text{tr}}$')
plt.xlabel('Position (µm)'); plt.ylabel('Trigger prob.')
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.title(f'Trigger probability (constant E = {E_mult:.2e} V/cm)')
plt.show()
print(f"Pe(0) = {Pe_tr[0]:.4f}, Ptr_avg = {np.mean(Pe_tr+Ph_tr-Pe_tr*Ph_tr):.4f}")

---## 9. Interactive explorationDrag sliders to see how multiplication layer width and field affect the gain.

In [ ]:
from ipywidgets import FloatSlider, interactive, VBox, HBox, Outputfrom IPython.display import displayW_slider = FloatSlider(value=0.5, min=0.1, max=1.5, step=0.05,                       description='W (µm):', continuous_update=False,                       style={'description_width': '80px'},                       layout={'width': '380px'})E_slider = FloatSlider(value=4.5e5, min=2.5e5, max=6.5e5, step=5e3,                       description='E (V/cm):', readback_format='.0e',                       continuous_update=False,                       style={'description_width': '80px'},                       layout={'width': '450px'})out = Output()@out.capture(clear_output=True)def plot_regions(W, E):    W_cm = W * 1e-4    alpha, beta = ionization_at(E)    M = mcintyre_M(alpha, beta, W_cm)        Es = np.linspace(2.5e5, 6.5e5, 200)    Ms = [mcintyre_M(*ionization_at(e), W_cm) for e in Es]        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))        ax1.semilogy(Es, Ms, 'b-', lw=2)    ax1.axvline(E, color='r', ls='--', alpha=0.5, label=f'E = {E:.2e} V/cm')    ax1.axhline(1, color='gray', ls=':')    ax1.set_xlabel('E (V/cm)')    ax1.set_ylabel('M')    ax1.grid(alpha=0.3)    ax1.legend()    ax1.set_title(f'M vs Field (W = {W:.2f} µm)')        Ws = np.linspace(0.05, 2.0, 100)    Ms_w = [mcintyre_M(*ionization_at(E), w*1e-4) for w in Ws]        ax2.semilogy(Ws, Ms_w, 'r-', lw=2)    ax2.axvline(W, color='b', ls='--', alpha=0.5, label=f'W = {W:.2f} µm')    ax2.axhline(1, color='gray', ls=':')    ax2.set_xlabel('W (µm)')    ax2.set_ylabel('M')    ax2.grid(alpha=0.3)    ax2.legend()    ax2.set_title(f'M vs Width (E = {E:.2e} V/cm)')        plt.tight_layout()    plt.show()        print(f"W = {W:.2f} µm, E = {E:.2e} V/cm")    print(f"alpha = {alpha:.3e} cm⁻¹, beta = {beta:.3e} cm⁻¹, k = {beta/(alpha+1e-30):.4f}")    print(f"M = {M:.2f}")widget = interactive(plot_regions, W=W_slider, E=E_slider)display(VBox([HBox(widget.children[:2]), out]))print("Drag sliders to explore the three-region model.")

---
## 8. Summary

| Quantity | Full sim | 3-region model |
|----------|----------|----------------|
| $\alpha, \beta$ | $\alpha(E(x))$, $\beta(E(x))$ | Constant per region |
| $M$ | McIntyre integral | $1/[1 - \beta(e^{\delta W} - 1)/\delta]$ |
| $P_{\text{tr}}$ | Coupled ODE BVP | Fixed-point on uniform grid |

The multiplication region alone ($W \approx 0.5$ µm, $E \approx 4.5\times10^5$ V/cm) determines almost all the avalanche behavior. The absorber contributes negligible multiplication — its role is photon absorption and drift.